# V5 ground-truth generation (all cases, HPC)

**WRITES TO DISK** (unlike `gt_diagnostics.ipynb`, which was evaluate-only). For every `predrr` knee
this generates the **V5 anatomy-aware solid bone-occupancy GT** and writes it twice:

1. **Portable** `data/interim/gt_v5/{healthy,fractured}/<case>_<Side>.nii.gz` (uint8 0/1, same affine
   as the source predrr) — inspectable in Slicer / 3D viewers.
2. **Decoder cache** `data/interim/predrr_occupancy_256/{dataset}_{case}_{side}.npy` (float32 256³) —
   the exact path + key `decoder_pipeline.load_gt_occupancy` reads as a cache hit, so training consumes
   V5 with **no decoder code change**. The `.npy` side is **lowercase** to match the DRR metadata key.

**V5** = per-component, (1) a size-capped morphological closing that closes open cortical rings without
welding across a joint/fracture gap, then (2) an adaptive size-capped multi-axis fill of the enclosed
canal. On the diagnostics subset: `bridging = 0` on every knee, honest holes ≈ 6.9% (gt_diagnostics §7).
Grounded in morphological closing + size-bounded acceptance (Soille; Vincent, IEEE TIP 1993).

**Run modes.** `ENV="local"` → dry-run the first `LIMIT` knees, `N_JOBS=1`. `ENV="HPC"` → all 71 knees,
`N_JOBS=os.cpu_count()` (threading backend; scipy morphology releases the GIL). CPU only — no GPU used.
**To revert** training to the baseline shell GT, delete `data/interim/predrr_occupancy_256/`.

In [1]:
# ===== CONFIG =====
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")                       # headless (HPC-safe)
import matplotlib.pyplot as plt
import joblib
from scipy.ndimage import (binary_fill_holes, binary_closing, label, find_objects, zoom)
from skimage.measure import marching_cubes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ---- environment (the only cell you normally edit) ----
ENV           = "local"        # "local" (dry-run a few cases, N_JOBS=1) | "HPC" (all knees, parallel)
EXPLICIT_ROOT = None           # set only if find_root fails. HPC workdir fallback:
                               #   "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2_24020059"
RES        = 256               # native predrr resolution == decoder HPC TARGET_RES
GT_THRESH  = 0.40              # bone threshold on the [0,1] windowed CT (same as decoder / gt_diagnostics)
LIMIT      = 3 if ENV == "local" else None              # dry-run: first LIMIT knees; HPC: None = all
N_JOBS     = 1 if ENV == "local" else (os.cpu_count() or 1)   # threading backend (scipy frees the GIL)

# ---- V5 knobs (identical to gt_diagnostics) ----
MIN_COMP_VOX    = 100
HOLE_MAX_VOX    = 1200
HOLE_MAX_FRAC   = 0.50
RING_CLOSE_ITER = 4
MAJOR_FRAC      = 0.01

# ---- project root + paths (find_root pattern from decoder_pipeline) ----
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in this cell.")

ROOT       = find_root(Path.cwd())
DATA       = ROOT / "data"
PREDRR_DIR = DATA / "interim" / "predrr"
OUT_NII    = DATA / "interim" / "gt_v5"                         # portable .nii.gz solids
OUT_NPY    = DATA / "interim" / ("predrr_occupancy_%d" % RES)   # decoder .npy occupancy cache (drop-in)
DRR_META   = DATA / "interim" / "DRRs" / "drr_generation_metadata.csv"
REPORT_DIR = ROOT / "reports" / "gt_v5"
for d in (OUT_NII / "healthy", OUT_NII / "fractured", OUT_NPY, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("ENV", ENV, "| ROOT:", ROOT)
print("N_JOBS:", N_JOBS, "| LIMIT:", LIMIT, "| RES:", RES, "| GT_THRESH:", GT_THRESH)
print("nii  ->", OUT_NII)
print("npy  ->", OUT_NPY)

ENV local | ROOT: C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject
N_JOBS: 1 | LIMIT: 3 | RES: 256 | GT_THRESH: 0.4
nii  -> C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\data\interim\gt_v5
npy  -> C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\data\interim\predrr_occupancy_256


## 1. Loaders + V5 builder (copied verbatim from `gt_diagnostics.ipynb`)

The morphology below is byte-identical to the validated V5 in the diagnostics notebook — do not edit
here; change it there and re-validate first.

In [2]:
# ---- path + load helpers (verbatim from gt_diagnostics) ----
def gt_file(dataset, case, side):
    Side = "Right" if str(side).lower().startswith("r") else "Left"
    if dataset == "healthy":
        return PREDRR_DIR / "healthy" / ("%s_%s.nii.gz" % (case, Side))
    return PREDRR_DIR / "fractured" / ("%s_Part%s.nii.gz" % (case, Side))

def load_continuous(dataset, case, side):
    return nib.load(str(gt_file(dataset, case, side))).get_fdata().astype(np.float32)

def binarize(vol, thr):
    return vol > thr

def all_knees():
    out = []
    for f in sorted((PREDRR_DIR / "healthy").glob("*.nii.gz")):
        case, Side = f.name[:-7].rsplit("_", 1); out.append(("healthy", case, Side.lower()))
    for f in sorted((PREDRR_DIR / "fractured").glob("*.nii.gz")):
        case, Side = f.name[:-7].split("_Part"); out.append(("fractured", case, Side.lower()))
    return out

# ---- V5 morphology (verbatim from gt_diagnostics cell d2f18f2a) ----
def _bbox_slices(binv, pad=0):
    '''Tight bounding-box slices around the True voxels (with optional padding), or None if empty.'''
    nz = np.nonzero(binv)
    if len(nz[0]) == 0:
        return None
    return tuple(slice(max(int(c.min()) - pad, 0), min(int(c.max()) + pad + 1, binv.shape[i]))
                 for i, c in enumerate(nz))

def clean_speckle(binv, min_vox=MIN_COMP_VOX):
    '''Drop connected components smaller than min_vox. Unlike keep_largest(K), this preserves real
    fracture fragments (sizeable) while removing threshold speckle (tiny) -- no fixed bone count.'''
    binv = np.asarray(binv, bool)
    lab, ncomp = label(binv)
    if ncomp == 0:
        return binv
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    keep = np.where(sizes >= min_vox)[0]
    return np.isin(lab, keep)

def _fill_along(binv, axis):
    '''Per-slice 2D hole fill along one axis (fills ALL enclosed holes, any size).'''
    out = np.zeros_like(binv, bool)
    for i in range(binv.shape[axis]):
        sl = np.take(binv, i, axis=axis)
        idx = [slice(None)] * 3; idx[axis] = i
        out[tuple(idx)] = binary_fill_holes(sl)
    return out

def _capped_fill_along(binv, axis, max_hole_vox):
    '''Per-slice 2D fill along one axis, but only fill an enclosed hole if it is SMALLER than
    max_hole_vox. Fills the narrow marrow canal while leaving large joint/fracture air gaps OPEN.'''
    out = np.array(binv, bool)
    for i in range(binv.shape[axis]):
        idx = [slice(None)] * 3; idx[axis] = i
        sl = binv[tuple(idx)]
        holes = binary_fill_holes(sl) & ~sl
        if holes.any():
            hl, hn = label(holes)
            sz = np.bincount(hl.ravel()); sz[0] = 0
            small = np.isin(hl, np.where((sz > 0) & (sz <= max_hole_vox))[0])
            o = out[tuple(idx)]; o |= small; out[tuple(idx)] = o
    return out

def _capped_multiaxis_fill(v, max_hole_vox):
    '''3D enclosed cavities (always safe) + size-capped 2D fill along all 3 axes (catches the canal
    whichever orientation encloses it, without filling large air gaps).'''
    out = binary_fill_holes(v)
    for ax in (0, 1, 2):
        out |= _capped_fill_along(v, ax, max_hole_vox)
    return out

def _capped_close(binv, iters, max_hole_vox):
    '''Size-capped morphological closing: binary_closing(iters), but ACCEPT only the added connected
    regions <= max_hole_vox. Closes an OPEN cortical ring (a small added bridge) so the marrow canal
    becomes enclosed and fillable, while a large joint/fracture-gap addition is REJECTED -> bridge-safe
    by construction. Grounded in morphological closing + size-bounded acceptance (Soille; Vincent 1993).'''
    binv = np.asarray(binv, bool)
    if iters <= 0:
        return binv
    closed = binary_closing(binv, iterations=iters)
    add = closed & ~binv
    if not add.any():
        return binv
    hl, hn = label(add)
    sz = np.bincount(hl.ravel()); sz[0] = 0
    small = np.isin(hl, np.where((sz > 0) & (sz <= max_hole_vox))[0])
    return binv | small

def solidify_per_component(binv, min_vox=MIN_COMP_VOX, ring_close_iter=RING_CLOSE_ITER,
                           max_hole_vox=HOLE_MAX_VOX, max_hole_frac=HOLE_MAX_FRAC):
    '''V5 (anatomy-aware): drop speckle, then solidify EACH component inside its own padded bbox.
    Per component: (1) a SIZE-CAPPED closing (_capped_close) closes open cortical rings so the canal
    becomes enclosed -- the cap rejects any large addition, so it never welds across a joint/fracture
    gap; (2) an ADAPTIVE size-capped multi-axis fill (cap = max(HOLE_MAX_VOX, frac * the bone's widest
    axial cross-section)) fills the now-enclosed canal. Per-component scope + both caps make bridging
    across distinct bones / fracture fragments impossible. (Soille; Vincent 1993.)'''
    clean = clean_speckle(binv, min_vox)
    lab, ncomp = label(clean)
    out = np.zeros(clean.shape, bool)
    P = max(ring_close_iter, 1) + 1
    for i, sl in enumerate(find_objects(lab), start=1):
        if sl is None:
            continue
        comp = np.pad(lab[sl] == i, P)               # pad so closing/erosion never shaves bbox faces
        axial_area_max = int(comp.sum(axis=(0, 1)).max()) if comp.any() else 0   # widest S-I cross-section
        cap = max(max_hole_vox, int(max_hole_frac * axial_area_max))             # adaptive hole cap
        comp = _capped_close(comp, ring_close_iter, cap)   # close open cortical rings (bridge-safe)
        comp = _capped_multiaxis_fill(comp, cap)           # fill the now-enclosed marrow canal
        out[sl] |= comp[P:-P, P:-P, P:-P]
    return out

def n_major_components(binv, min_frac=MAJOR_FRAC):
    '''Count components whose size >= min_frac of the bone volume (ignores speckle).'''
    binv = np.asarray(binv, bool); tot = binv.sum()
    if tot == 0:
        return 0
    lab, ncomp = label(binv)
    sz = np.bincount(lab.ravel()); sz[0] = 0
    return int((sz >= min_frac * tot).sum())

def bridging_count(variant, base):
    '''#major structures FUSED vs the cleaned baseline shell. >0 => distinct bones/fragments welded
    (the dangerous fracture artifact). Floored at 0.'''
    base_clean = clean_speckle(np.asarray(base, bool))
    return max(n_major_components(base_clean) - n_major_components(variant), 0)

print("V5 builder ready. knees on disk:", len(all_knees()))

V5 builder ready. knees on disk: 71


## 2. Generate one knee → write `.nii.gz` + `.npy`

In [3]:
def generate_knee(dataset, case, side):
    '''Build V5 solid occupancy for one knee and write both outputs. `side` is lowercase
    ("left"/"right"). The .npy cache key stays lowercase to match the decoder lookup; the .nii.gz
    mirrors the predrr filename (Left/Right) and keeps the source affine for spatial alignment.'''
    t0 = time.time()
    src = nib.load(str(gt_file(dataset, case, side)))          # keep image for its affine
    vol = src.get_fdata().astype(np.float32)
    base = binarize(vol, GT_THRESH)
    gt = solidify_per_component(base)                          # V5
    br = bridging_count(gt, base)                              # expect 0
    Side = "Right" if side.startswith("r") else "Left"
    name = ("%s_%s" % (case, Side)) if dataset == "healthy" else ("%s_Part%s" % (case, Side))
    nii_path = OUT_NII / dataset / (name + ".nii.gz")
    npy_path = OUT_NPY / ("%s_%s_%s.npy" % (dataset, case, side))   # lowercase side = decoder cache key
    nib.save(nib.Nifti1Image(gt.astype(np.uint8), src.affine), str(nii_path))
    np.save(npy_path, gt.astype(np.float32))
    return dict(dataset=dataset, case=case, side=side,
                occ_pct=100.0 * float(gt.mean()), n_vox=int(gt.sum()),
                n_components=int(n_major_components(gt)), bridging=int(br),
                seconds=round(time.time() - t0, 1),
                nii=str(nii_path.relative_to(ROOT)), npy=str(npy_path.relative_to(ROOT)))

print("generate_knee ready")

generate_knee ready


## 3. Batch driver (always overwrite) → manifest CSV

In [4]:
knees = all_knees()
if LIMIT:
    knees = knees[:LIMIT]
print("generating V5 GT for %d knees (overwrite)%s ..."
      % (len(knees), "" if not LIMIT else "  [DRY-RUN LIMIT=%d]" % LIMIT))
t0 = time.time()
if N_JOBS and N_JOBS != 1:
    rows = joblib.Parallel(n_jobs=N_JOBS, backend="threading")(
        joblib.delayed(generate_knee)(*k) for k in knees)
else:
    rows = []
    for k in knees:
        rows.append(generate_knee(*k))
        r = rows[-1]
        print("  %-9s %-10s %-5s occ=%5.2f%%  comp=%d  bridge=%d  (%.1fs)"
              % (k[0], k[1], k[2], r["occ_pct"], r["n_components"], r["bridging"], r["seconds"]))

manifest = pd.DataFrame(rows)
man_path = REPORT_DIR / "manifest.csv"
manifest.to_csv(man_path, index=False)
print("\nwrote %d knees in %.0fs -> %s" % (len(manifest), time.time() - t0, man_path))
print(manifest[["dataset", "case", "side", "occ_pct", "n_components", "bridging", "seconds"]]
      .round(2).to_string(index=False))

generating V5 GT for 3 knees (overwrite)  [DRY-RUN LIMIT=3] ...
  healthy   VSD_001    left  occ= 4.77%  comp=3  bridge=0  (11.1s)
  healthy   VSD_001    right occ= 5.08%  comp=4  bridge=0  (12.3s)
  healthy   VSD_002    left  occ= 3.71%  comp=2  bridge=0  (13.8s)

wrote 3 knees in 37s -> C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\reports\gt_v5\manifest.csv
dataset    case  side  occ_pct  n_components  bridging  seconds
healthy VSD_001  left     4.77             3         0     11.1
healthy VSD_001 right     5.08             4         0     12.3
healthy VSD_002  left     3.71             2         0     13.8


## 4. QC — bridging, occupancy, and decoder cache coverage

The **cache-coverage** check is the important one: every `(dataset, case, side)` the decoder will
request (from the DRR metadata) must have a matching `predrr_occupancy_256/*.npy`, otherwise
`load_gt_occupancy` silently re-thresholds and training falls back to the baseline shell. It is only
meaningful on a full run (`LIMIT=None`).

In [5]:
# QC 1: V5 must never weld distinct structures
bad = manifest[manifest.bridging > 0]
print("QC bridging : %d knee(s) with bridging>0%s"
      % (len(bad), "" if bad.empty else "   <-- INVESTIGATE"))
if not bad.empty:
    print(bad[["dataset", "case", "side", "bridging"]].to_string(index=False))

# QC 2: occupancy sanity (a solid knee is a few % of the 256^3 volume)
print("QC occupancy: min=%.2f%%  mean=%.2f%%  max=%.2f%%"
      % (manifest.occ_pct.min(), manifest.occ_pct.mean(), manifest.occ_pct.max()))

# QC 3: decoder cache coverage vs the DRR metadata (lowercase side key)
if DRR_META.exists():
    meta = pd.read_csv(DRR_META)[["dataset", "case", "side"]].drop_duplicates()
    meta["side"] = meta["side"].astype(str).str.lower()
    missing = [(r.dataset, r.case, r.side) for r in meta.itertuples(index=False)
               if not (OUT_NPY / ("%s_%s_%s.npy" % (r.dataset, r.case, r.side))).exists()]
    if LIMIT:
        print("QC cache-cov: SKIPPED (dry-run LIMIT=%s; run with LIMIT=None for the full check)" % LIMIT)
    else:
        print("QC cache-cov: %d/%d DRR knees have a V5 cache  | missing: %s"
              % (len(meta) - len(missing), len(meta), missing if missing else "none"))
else:
    print("QC cache-cov: DRR metadata not found at", DRR_META)

QC bridging : 0 knee(s) with bridging>0
QC occupancy: min=3.71%  mean=4.52%  max=5.08%
QC cache-cov: SKIPPED (dry-run LIMIT=3; run with LIMIT=None for the full check)


## 5. Visual QC — per-knee 3D surface + triplane with inline diagnostics (all generated knees)

Saves **one PNG per knee** to `reports/gt_v5/qc_per_knee/`. Reads from the `.npy` cache —
no re-running solidification. Five panels per figure:

- **Panel 1** — marching-cubes **3D surface** (blue, downsampled to 128³).
- **Panels 2–4** — mid-plane grayscale slices (L-R, A-P, S-I). Each panel title shows the
  **per-view occupancy** (% of non-zero pixels in that 2D cross-section), so over-fill or
  near-empty anatomy is immediately visible in the orientation where it occurs.
- **Panel 5** — **QC stats box**: global occ, n_vox, comp, fill +Δ, and bridge status,
  colour-matched to the suptitle (green = safe, red = bridging detected).

Suptitle: **green** when `bridge=0`, **red** when `bridge>0`.

| Field | Meaning | Healthy expectation |
| --- | --- | --- |
| per-view occ | % of that 2D slice that is bone | varies by slice position; spikes flag over-fill |
| `occ (global)` | V5 occupancy = % of the full 256³ volume that is bone | ~2–10 % |
| `n_vox` | absolute solid voxel count | — |
| `comp` | major connected components (≥1 % of bone volume) | matches expected #bones/fragments |
| `fill +Δ` | voxels V5 added over the raw `vol>0.40` shell, as % of the shell | small + for canal/ring fill; very large ⇒ possible over-fill |
| `bridge` | structures welded vs the cleaned baseline — **must be 0** | 0 (title/box turns **red** if >0) |

In [6]:
# Visual QC: per-knee marching-cubes surface + triplane (per-view occ) + inline stats box.
# One PNG per knee -> reports/gt_v5/qc_per_knee/. Reads from .npy cache (no re-solidification).
QC_DIR = REPORT_DIR / "qc_per_knee"
QC_DIR.mkdir(parents=True, exist_ok=True)

SURF_RES = 128   # volume downsampled to this before marching cubes (speed vs detail)

def _surface(ax, vol, color):
    '''Marching-cubes 3D surface, downsampled to SURF_RES for render speed.'''
    v = np.asarray(vol, np.float32)
    if (v > 0.5).sum() < 10:
        ax.text2D(0.5, 0.5, "(empty)", ha="center", transform=ax.transAxes); ax.set_axis_off(); return
    if v.shape[0] > SURF_RES:
        v = zoom(v, SURF_RES / v.shape[0], order=1)
    try:
        level = 0.5
        if v.min() >= level or v.max() <= level:
            level = float(v.min()) + 0.5 * (float(v.max()) - float(v.min()))
        verts, faces, _, _ = marching_cubes(v, level=level)
        mesh = Poly3DCollection(verts[faces], alpha=0.6); mesh.set_facecolor(color)
        ax.add_collection3d(mesh)
        ax.set_xlim(0, v.shape[0]); ax.set_ylim(0, v.shape[1]); ax.set_zlim(0, v.shape[2])
        ax.view_init(elev=15, azim=-70)
    except Exception as e:
        ax.text2D(0.5, 0.5, "MC failed:\n%s" % type(e).__name__, ha="center", transform=ax.transAxes)
    ax.set_axis_off()

def _qc_knee(row):
    '''5-panel per-knee figure: 3D surface | L-R mid | A-P mid | S-I mid | QC stats box.
    Each slice panel shows its own 2D occ% so over-fill or near-empty anatomy is localised by
    orientation. Stats box repeats the global metrics with bridge status in green/red.'''
    npy_path = OUT_NPY / ("%s_%s_%s.npy" % (row.dataset, row.case, row.side))
    if not npy_path.exists():
        print("  SKIP (npy missing):", npy_path.name); return
    gt        = np.load(npy_path).astype(bool)
    base      = binarize(load_continuous(row.dataset, row.case, row.side), GT_THRESH)
    base_vox  = int(base.sum())
    fill_gain = 100.0 * (int(row.n_vox) - base_vox) / base_vox if base_vox else 0.0
    mid       = [s // 2 for s in gt.shape]
    tc        = "darkgreen" if row.bridging == 0 else "red"
    bridge_tag = "bridge: 0  ✓ SAFE" if row.bridging == 0 else "bridge: %d  ✗ BRIDGING" % row.bridging

    fig = plt.figure(figsize=(17, 3.8))

    # Panel 1: 3D surface
    _surface(fig.add_subplot(1, 5, 1, projection="3d"), gt, "tab:blue")

    # Panels 2-4: mid-plane slices with per-view occ in the panel title
    axis_labels = ("L-R", "A-P", "S-I")
    for col_i, axis in enumerate((0, 1, 2)):
        sl = np.take(gt.astype(np.float32), mid[axis], axis=axis)
        slice_occ = 100.0 * float(sl.mean())
        ax = fig.add_subplot(1, 5, 2 + col_i)
        ax.imshow(sl, cmap="gray", vmin=0, vmax=1)
        ax.set_title("%s mid\nocc %.1f%%" % (axis_labels[col_i], slice_occ), fontsize=8)
        ax.axis("off")

    # Panel 5: QC stats text box (global metrics + bridge status)
    ax_s = fig.add_subplot(1, 5, 5)
    ax_s.axis("off")
    stats_lines = [
        "%s  %s  %s" % (row.dataset, row.case, row.side),
        "",
        "occ (global)  %.2f%%" % float(row.occ_pct),
        "n_vox         %d"     % int(row.n_vox),
        "comp          %d"     % int(row.n_components),
        "fill +Δ       +%.1f%%" % fill_gain,
        "",
        bridge_tag,
    ]
    ax_s.text(0.05, 0.95, "\n".join(stats_lines), transform=ax_s.transAxes,
              fontsize=8, va="top", family="monospace", color=tc,
              bbox=dict(boxstyle="round,pad=0.4", facecolor="whitesmoke", edgecolor=tc, lw=1.2))

    fig.suptitle(
        "%s  %s  %s   |   occ=%.2f%%   comp=%d   fill +%.1f%%   %s"
        % (row.dataset, row.case, row.side, float(row.occ_pct),
           int(row.n_components), fill_gain, bridge_tag),
        fontsize=10, color=tc)
    plt.tight_layout()
    fig.savefig(str(QC_DIR / ("qc_%s_%s_%s.png" % (row.dataset, row.case, row.side))),
                dpi=100, bbox_inches="tight"); plt.close(fig)

t0_qc = time.time()
for row in manifest.itertuples(index=False):
    _qc_knee(row)
print("saved %d per-knee QC images in %.0fs  ->  %s" % (len(manifest), time.time() - t0_qc, QC_DIR))

saved 3 per-knee QC images in 48s  ->  C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\reports\gt_v5\qc_per_knee
